This part of the pipeline clusters all BGCs found for this entire genome dataset using BiG-SCAPE.

### Paths and parameters

#### Pipeline input folders

In [ ]:
antismash_output="09-MGEs/BGCs/output"
classification='02-GTDB/classification_table'

#### Pipeline output folders

In [ ]:
task_root="10-BGCClustering"
bigscape_input="$task_root/input"
bigscape_output="$task_root/output"
network_folder=$(find "$bigscape_output/network_files" -type d | grep -E 'mix$')

mkdir -p $task_root $bigscape_input $bigscape_output

#### Tool pointers and parameters

In [ ]:
pfam_db="/mnt/STORAGE/databases/PFAM"
n_cores=22
cutoffs="0.2 0.25 0.3 0.35 0.4 0.45 0.50 0.55 0.6 0.65 0.7"

annotate_network="utils/annotate_bigscape_network.py"

### Checking dependencies

In [ ]:
conda activate bigscape
bigscape --version
conda deactivate

### Gathering all antiSMASH region genbank files

Copy the antiSMASH region GenBank file directory structure into the BiG-SCAPE input folder, and then collapse it by pulling all GenBanks out of their folder.

In [ ]:
find $antismash_output | grep -E \.region[0-9]{3}\.gbk | xargs -I % bash -c \
'ln -s -T $(pwd)/% $(pwd)/10-BGCClustering/input/$(echo % | cut -d "/" -f 4- | tr "/" ".")'

### Running BiG-SCAPE

In [ ]:
conda activate bigscape

In [ ]:
bigscape -i $bigscape_input -o $bigscape_output --pfam_dir $pfam_db -c $n_cores --include_singletons --cutoffs $cutoffs --mix --mibig

In [ ]:
conda deactivate

### Annotate network files

Add order metadata and duplicate the links so that the network can be easily imported into CytoScape.

In [ ]:
dir -1 $network_folder | grep -E "network$" | xargs -I % \
python $annotate_network $network_folder/% $classification 'order' $antismash_output $task_root/%.annotated

**Ready for visualisation in CytoScape!**